# DATA301 Project — LSH-Accelerated Collaborative Filtering for Movie Recommendations

**Research Question:** What is the difference in movie recommendation accuracy and computational runtime between LSH-accelerated collaborative filtering and brute-force collaborative filtering when applied to the Amazon Movies and TV reviews dataset?

**Pipeline Overview:**
1. Load and preprocess the Amazon Movies and TV dataset using Dask
2. Build item-to-user sets from positive ratings
3. Compute MinHash signatures using Dask Bag .map() for parallel processing
4. Apply LSH banding to identify candidate similar item pairs
5. Compute cosine similarity within candidate pairs (LSH approach) vs all pairs (brute-force)
6. Generate top-10 recommendations using item-based collaborative filtering
7. Evaluate using Precision@10, RMSE, and MAE
8. Measure scalability on single processor and Google Cloud




---
## Section 1 — Setup

This notebook saves intermediate train/test files locally in the `data/` folder so preprocessing does not need to be repeated.


In [ ]:
# Set up output DIR so the train and test files can be saved
SAVE_DIR = "./data"

import os

os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Saving files to: {SAVE_DIR}")

In [ ]:
# SESSION RELOAD - run this if session reset
import pandas as pd
import numpy as np
from collections import defaultdict
import time


train_path = f"{SAVE_DIR}/train.csv"
test_path  = f"{SAVE_DIR}/test.csv"

if not os.path.exists(train_path):
    print("No saved CSVs found in Drive — run the full pipeline from the download cell instead")
else:
    print("Loading train/test splits from Google Drive...")
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    print(f"Train: {len(train_df):,} ratings | Test: {len(test_df):,} ratings")
    print("Done — skip straight to the Building Sets cell")

In [ ]:
# install daskbah
!pip install dask[bag] --quiet

In [ ]:
# Imports
import dask.bag as db
import dask.dataframe as df
import pandas as pd
import numpy as np
import json
import gzip
from collections import defaultdict
import time
import os
import matplotlib.pyplot as plt

print("All imports done")

---
## Section 2 — Data Loading and Preprocessing

The dataset used is the Amazon Movies and TV Reviews (5-core) from the UC San Diego McAuley Lab dataset collection (McAuley, 2023). The 5-core subset ensures every user and item has at least 5 interactions.

Each record contains:
- reviewerID → user id
- asin → item (movie) id
- overall → rating (1–5)


In [ ]:
# Download the dataset
!wget -q -O reviews_Movies_and_TV_5.json.gz \
    "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz"

print("Download complete")
print(f"File size: {os.path.getsize('reviews_Movies_and_TV_5.json.gz') / 1e6:.1f} MB")

In [ ]:
# inspect the raw data
with gzip.open("reviews_Movies_and_TV_5.json.gz", "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(json.loads(line))
        if i >= 2:  # just print first 3 records
            break

In [ ]:
# Parse the raw JSON using dask bag
def parse_record(line):
  """Parse the raw JSON and extract only the fields we need"""
  record = json.loads(line)
  return {
      "userId": record.get("reviewerID"),
      "itemId": record.get("asin"),
      "rating": float(record.get("overall"))
  }

raw_bag = db.read_text("reviews_Movies_and_TV_5.json.gz", encoding="utf-8")

# parse each line into the clean dict
parsed_bag = raw_bag.map(parse_record)

# get the first three records
print("Sample records:")
for record in parsed_bag.take(3):
    print(record)

In [ ]:
# Convert the dask bag to a dask dataframe
df = parsed_bag.to_dataframe(meta={
    "userId": str,
    "itemId": str,
    "rating": float
})

# drop the rows with missing values
df = df.dropna()

# compute some basic stats
print("Computing some dataset stats (this takes a moment)...")
num_reviews  = len(df)
num_users = df["userId"].nunique().compute()
num_items = df["itemId"].nunique().compute()
avg_rating = df["rating"].mean().compute()

print(f"\nDataset loaded successfully:")
print(f"  Total reviews : {num_reviews:,}")
print(f"  Unique users  : {num_users:,}")
print(f"  Unique items  : {num_items:,}")
print(f"  Average rating: {avg_rating:.2f}")

### Train / Test Split

Split the data 80/20 per user; holding out 20% of each user's ratings as the test set. This ensures every test user also has training history, which is required for collaborative filtering.


In [ ]:
# Split into train and test split
print("Spliting data into splits (this takes a moment)...")
print("Loading into memory...")
df_pd = df.compute()

np.random.seed(42)

def split_user(group):
  """Shuffle and then assign 20% to test and 80% to train for each user"""
  group = group.sample(frac=1, random_state=42)
  n_test = max(1, int(len(group) * 0.2))
  group["split"] = "train"
  group.iloc[:n_test, group.columns.get_loc("split")] = "test"
  return group

print("Splitting data...")
df_split = df_pd.groupby("userId", group_keys=False).apply(split_user)

train_df = df_split[df_split["split"] == "train"].drop(columns="split").reset_index(drop=True)
test_df  = df_split[df_split["split"] == "test"].drop(columns="split").reset_index(drop=True)

print(f"\nTrain set: {len(train_df):,} ratings")
print(f"Test set : {len(test_df):,} ratings")
print(f"Split    : {len(train_df)/(len(train_df)+len(test_df))*100:.1f}% / {len(test_df)/(len(train_df)+len(test_df))*100:.1f}%")

# save to google drive so they are persistent
train_df.to_csv(f"{SAVE_DIR}/train.csv", index=False)
test_df.to_csv(f"{SAVE_DIR}/test.csv",  index=False)
print(f"\nSaved to Google Drive: {SAVE_DIR}")

---
## Section 3 — Building Item-to-User Sets

For MinHash, we represent each movie as the set of users who rated it 3 stars or above. The Jaccard similarity between two of these sets estimates how similar two movies are.


In [ ]:
# Build the item to user sets from the training data
print("Building user sets for highly rated items")

# only positive ratings for similarity
positive = train_df[train_df["rating"] >= 3.0]

# group by the movies
item_user_sets = (
    positive
    .groupby("itemId")["userId"]
    .apply(set)
    .to_dict()
)

#filter out all the movies that dont have at least 5 positive ratings
item_user_sets = {k: v for k, v in item_user_sets.items() if len(v) >= 5}

print(f"Items with 5+ positive ratings: {len(item_user_sets):,}")

# check an example
example_item = list(item_user_sets.keys())[0]
print(f"\nExample item: {example_item}")
print(f"  Liked by {len(item_user_sets[example_item])} users")

In [ ]:
# Distribution graph of ratings per item
import matplotlib.pyplot as plt
import numpy as np

# count ratings per item from training data
ratings_per_item = train_df.groupby("itemId")["rating"].count().sort_values(ascending=False).reset_index()
ratings_per_item.columns = ["itemId", "count"]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
  range(len(ratings_per_item)),
  ratings_per_item["count"].values,
  color="blue",
  linewidth=1
)

ax.set_xlabel("Most to least rated items", fontsize=11)
ax.set_ylabel("Number of Ratings", fontsize=11)
ax.set_title("Long-tail Distribution of Amazon Movies and TV Reviews", fontsize=13)
ax.grid(True, alpha=0.3)

# highlight under the graph
ax.fill_between(
  range(len(ratings_per_item)),
  ratings_per_item["count"].values,
  color="blue",
  alpha=0.2
)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/long_tail_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# OPTIONAL

### The cell below was run for further testing to try decrease sparcity. Shoudnt be run in a standard full pipeline run and has therefore been commented out.

In [ ]:
# # OPTIONAL - further filtering tests
# # Run this cell to test with the dataset filtered further to only include specific values
# # All cells below will use whichever item_user_sets is currently in memory

# FILTER_THRESHOLD = 10 # minimun ratings required for both users and items

# # filter users with at least FILTER_THRESHOLD ratings
# user_counts = train_df.groupby("userId")["itemId"].count()
# valid_users = set(user_counts[user_counts >= FILTER_THRESHOLD].index)

# # filters items with at least FILTER_THRESHOLD ratings
# item_counts = train_df.groupby("itemId")["userId"].count()
# valid_items = set(item_counts[item_counts >= FILTER_THRESHOLD].index)

# # apply both filters to training data
# filtered_train = train_df[
#     train_df["userId"].isin(valid_users) &
#     train_df["itemId"].isin(valid_items)
# ]

# # rebuild item_user_sets from filtered data
# positive_filtered = filtered_train[filtered_train["rating"] >= 3.0]
# item_user_sets = (
#     positive_filtered
#     .groupby("itemId")["userId"]
#     .apply(set)
#     .to_dict()
# )
# item_user_sets = {k: v for k, v in item_user_sets.items() if len(v) >= 5}

# # also rebuild rating vectors and user lookups from filtered data
# # so the pipeline uses consistent data throughout
# item_rating_vectors = (
#     filtered_train
#     .groupby("itemId")
#     .apply(lambda g: dict(zip(g["userId"], g["rating"])))
#     .to_dict()
# )
# user_train_items = (
#     filtered_train
#     .groupby("userId")
#     .apply(lambda g: dict(zip(g["itemId"], g["rating"])))
#     .to_dict()
# )

# print(f"Filtering applied (threshold = {FILTER_THRESHOLD} ratings)")
# print(f"Items remaining : {len(item_user_sets):,} (was ~40,000)")
# print(f"Users remaining : {len(valid_users):,}")
# print(f"Ratings remaining: {len(filtered_train):,}")

---
## Section 4 — MinHash Signature Generation

MinHash compresses each item's user set into a compact **signature** — a short array of integers.

This means that items with similar user sets will mostly have similar signatures, without needing to compare the full sets directly.

Each hash function takes the form h(x) = (ax + b) mod p

In [ ]:
# Set up the minhash
# we use a large prime p to spread the values across a wide range
# map the user IDs to integers (hash functions need numeric input)

print("Mapping user IDs to integers...")
all_users = list({u for users in item_user_sets.values() for u in users})
user_to_int = {user: index for index, user in enumerate(all_users)}
num_users_total = len(user_to_int)
print(f"Total unique users in positive ratings: {num_users_total:,}")

def make_hash_params(num_hashes, seed=42):
  """Generate random (a, b) parameter pairs for num_hashes hash functions,
  These are the random vraiables for h(x) = (ax+b) mod p"""
  np.random.seed(seed)
  p = 2**31 - 1 # large prime
  a = np.random.randint(1, p, size=num_hashes).astype(np.int64)
  b = np.random.randint(0, p, size=num_hashes).astype(np.int64)
  return a, b, p

def minhash_signature(user_set, a, b, p, user_to_int):
  """Compute MinHash signature for one items set of users.
  Uses numpy to compute all hash functions at the same time."""
  p_val = int(p)

  # convert userIds to integer array
  user_ints = np.array(
    [user_to_int[u] for u in user_set if u in user_to_int],
    dtype=np.int64
  )
  if len(user_ints) == 0:
    return np.full(len(a), p_val, dtype=np.int64)
  # compute all hash values at once using broadcasting
  hash_matrix = (np.outer(a, user_ints) + b[:, None]) % p_val
  # take the minimun accross users for each hash function
  return hash_matrix.min(axis=1)

def run_minhash(NUM_HASHES, item_sets=None):
  """Compute the minhash signatures for all items and return the signature
  dict and the time taken"""
  if item_sets is None:
    item_sets = item_user_sets

  a, b, p = make_hash_params(NUM_HASHES)
  items_bag = db.from_sequence(list(item_sets.items()), npartitions=8)

  # capture a, b, p as default arguments to avoid issues within the loop
  def compute_signatures(item_tuple, _a=a, _b=b, _p=p):
    item_id, user_set = item_tuple
    return (item_id, minhash_signature(user_set, _a, _b, _p, user_to_int))

  start = time.time()
  signatures = dict(items_bag.map(compute_signatures).compute())
  elapsed = time.time() - start

  print(f"  MinHash    : {len(signatures):,} signatures in {elapsed:.1f}s")
  return signatures, elapsed

print("MinHash functions ready")

---
## Section 5 — Locality Sensitive Hashing (LSH) Banding

LSH banding solves the scalability problem of collaborative filtering. Computing cosine similarity between **all pairs** of 40,000+ items would result in over 800 million comparisons. LSH reduces this dramatically.

How the banding works:
1. Each item's MinHash signature is split into b bands of r rows
2. Items with signatures that agree on any and at least one band are placed in the same candidate bucket
3. Items sharing a bucket become candidate pairs for the cosine similarity step

ROWS_PER_BAND = 2 is kept fixed across all runs (50, 100, 200 hashes) so that changing NUM_HASHES only affects signature length.


In [ ]:
# LSH Banding function

def run_lsh_banding(signatures, NUM_HASHES):
  """Split signatures into bands and group items into candidate buckets.
  Returns candidate_pairs set and NUM_BANDS used.
  """
  ROWS_PER_BAND = 2
  NUM_BANDS = NUM_HASHES // ROWS_PER_BAND

  def get_band_keys(item_tuple):
    """For one item generate all (band_key, item_id) pairs — runs in parallel"""
    item_id, sig = item_tuple
    pairs = []
    for band_index in range(NUM_BANDS):
      s = band_index * ROWS_PER_BAND
      band_slice = tuple(int(x) for x in sig[s:s + ROWS_PER_BAND])
      # string key as there was an issue with the tuples
      key = f"{band_index}_{band_slice[0]}_{band_slice[1]}"
      pairs.append((key, item_id))
    return tuple(pairs)

  # map each item to its band keys in parallel
  sigs_bag = db.from_sequence(list(signatures.items()), npartitions=8)
  band_pairs = sigs_bag.map(get_band_keys).flatten()

  # group by band key to form buckets
  grouped = band_pairs.groupby(lambda x: x[0])

  # step 3 — extract candidate pairs from each bucket in parallel
  def extract_pairs(bucket_tuple):
    """Extract all candidate pairs from one bucket"""
    _, items_in_bucket = bucket_tuple
    item_ids = [item_id for _, item_id in items_in_bucket]
    pairs = []
    if len(item_ids) > 1:
      for i in range(len(item_ids)):
        for j in range(i + 1, len(item_ids)):
          pairs.append(tuple(sorted([item_ids[i], item_ids[j]])))
    return pairs

  candidate_pairs = set(
      grouped
      .map(extract_pairs)
      .flatten()
      .compute(scheduler='synchronous')
    )

  total_items = len(signatures)
  brute_force_pairs = total_items * (total_items - 1) // 2

  print(f"  LSH        : {len(candidate_pairs):,} candidate pairs "
        f"({NUM_BANDS} bands x {ROWS_PER_BAND} rows)")
  print(f"  Reduction  : {100 * (1 - len(candidate_pairs)/brute_force_pairs):.1f}% "
        f"fewer comparisons than brute-force")

  return candidate_pairs, NUM_BANDS

print("LSH banding function ready")


---
## Section 6 — Single Processor Scalability

We measure how the LSH pipeline runtime scales with problem size on a single processor. The pipeline (MinHash and LSH banding) is run at 25%, 50%, 75%, and 100% of the full item list and the runtime is recorded at each size.


In [ ]:
# Single Processor Scalability Test

def run_scalability_test(NUM_HASHES):
  """Run minhash and lsh banding at 4 problem sizes and return runtimes"""
  ROWS_PER_BAND = 2
  NUM_BANDS = NUM_HASHES // ROWS_PER_BAND
  sizes = [0.25, 0.5, 0.75, 1.0]
  all_items = list(item_user_sets.items())
  a_s, b_s, p_s = make_hash_params(NUM_HASHES)
  item_counts = []
  runtimes = []

  print(f"  NUM_HASHES={NUM_HASHES}:")
  for size in sizes:
    n = int(len(all_items) * size)
    subset = dict(all_items[:n])

    # Use Dask Bag for MinHash
    subset_bag = db.from_sequence(list(subset.items()), npartitions=8)
    start = time.time()
    subset_signatures = dict(subset_bag.map(
        lambda x: (x[0], minhash_signature(x[1], a_s, b_s, p_s, user_to_int))
    ).compute())

    # LSH banding
    buckets = defaultdict(list)
    for item_id, signature in subset_signatures.items():
      for band_index in range(NUM_BANDS):
        s = band_index * ROWS_PER_BAND
        key = (band_index, tuple(signature[s:s + ROWS_PER_BAND]))
        buckets[key].append(item_id)
    pairs = set()
    for bucket in buckets.values():
      if len(bucket) > 1:
        for i in range(len(bucket)):
          for j in range(i + 1, len(bucket)):
            pairs.add(tuple(sorted([bucket[i], bucket[j]])))

    elapsed = time.time() - start
    item_counts.append(n)
    runtimes.append(elapsed)
    print(f"    {int(size*100)}%  {n:>10,}  {elapsed:.1f}s")

  return item_counts, runtimes

print("Scalability test function ready")


In [ ]:
# PLot the results from the single processor scalability
# Compare the actual runtime to the ideal linear scaling
def plot_scalability(scalability_results):
  """Plot scalability results for multiple NUM_HASHES configurations"""
  fig, ax = plt.subplots(figsize=(8, 5))
  colors = {50: 'green', 100: 'orange', 200: 'blue'}

  for num_hashes, (sizes, times) in scalability_results.items():
    ax.plot(sizes, times,
            'o-', color=colors[num_hashes], linewidth=2, markersize=8,
            label=f'{num_hashes} hash functions')

  # ideal linear scaling reference based on the 200-hash
  base_sizes, base_times = scalability_results[200]
  ideal = [base_times[0] * (s / base_sizes[0]) for s in base_sizes]
  ax.plot(base_sizes, ideal,
          '--', color='gray', linewidth=1.5, label='Ideal linear scaling')

  ax.set_xlabel('Number of Items')
  ax.set_ylabel('Runtime (seconds)')
  ax.set_title('Single Processor Scalability — LSH Pipeline', fontsize=13)
  ax.legend(fontsize=11)
  ax.grid(True, alpha=0.3)

  plt.tight_layout()
  plt.savefig(f"{SAVE_DIR}/scalability_single_processor.png", dpi=150, bbox_inches='tight')
  plt.show()
  print(f"Plot saved to: {SAVE_DIR}/scalability_single_processor.png")

print("Scalability plot function ready")

---
## Section 7 — Collaborative Filtering

With candidate pairs identified, we now compute the cosine similarity between them to build the item-item similarity set. This is the collaborative filtering.

Cosine similarity between two items A and B is computed from their rating vectors (the full set of user ratings, not just positive ones):

    sim(A, B) = dot(A, B) / (|A| × |B|)

For LSH, we only compute similarity for the candidate pairs found by LSH. For brute-force, we extrapolate the full runtime from a timed sample.


In [ ]:
# Build item rating vectors and user lookup tables
# These are computed outside the loop since they don't change between iterations

print("Building item rating vectors from training data...")

item_rating_vectors = (
    train_df
    .groupby("itemId")
    .apply(lambda g: dict(zip(g["userId"], g["rating"])), include_groups=False)
    .to_dict()
)

print(f"Rating vectors built for {len(item_rating_vectors):,} items")

# build a user -> training items lookup for recommendation phase
print("Building user training history...")
user_train_items = (
    train_df
    .groupby("userId")
    .apply(lambda g: dict(zip(g["itemId"], g["rating"])), include_groups=False)
    .to_dict()
)

# build user -> test items lookup for evaluation
user_test_items = (
    test_df
    .groupby("userId")
    .apply(lambda g: dict(zip(g["itemId"], g["rating"])), include_groups=False)
    .to_dict()
)

print(f"Training history built for {len(user_train_items):,} users")
print(f"Test history built for    {len(user_test_items):,} users")

In [ ]:
# Cosine similarity function and run_similarity function

def cosine_similarity(item_a, item_b, rating_vectors):
  """Compute cosine similarity between two items based on user ratings.
  Returns a value between 0 (no similarity) and 1 (identical rating patterns).
  This is the most computational heacy process in pipeline"""
  vec_a = rating_vectors.get(item_a, {})
  vec_b = rating_vectors.get(item_b, {})

  # only users who rated both items matter
  common_users = set(vec_a.keys()) & set(vec_b.keys())
  if not common_users:
    return 0.0

  # get the dot product of ratings for common users
  dot_product = sum(vec_a[u] * vec_b[u] for u in common_users)

  # find the magnitudes using all ratings
  mag_a = np.sqrt(sum(v ** 2 for v in vec_a.values()))
  mag_b = np.sqrt(sum(v ** 2 for v in vec_b.values()))

  if mag_a == 0 or mag_b == 0:
    return 0.0

  return dot_product / (mag_a * mag_b)

def run_similarity(candidate_pairs, rating_vectors=None):
  """Compute cosine similarity for all lsh candidate pairs
  Returns the similarity index and the time taken."""
  if rating_vectors is None:
    rating_vectors = item_rating_vectors

  start_lsh = time.time()
  lsh_similarities = defaultdict(dict)

  for item_a, item_b in candidate_pairs:
    sim = cosine_similarity(item_a, item_b, rating_vectors)
    if sim > 0:
      lsh_similarities[item_a][item_b] = sim
      lsh_similarities[item_b][item_a] = sim

  lsh_time = time.time() - start_lsh
  print(f"  Similarity : index built in {lsh_time:.1f}s "
        f"({len(lsh_similarities):,} items with neighbours)")
  return lsh_similarities, lsh_time

print("Similarity functions ready")

---
## Section 8 — Recommendation Generation

With the item-item similarity index built, we generate top-10 recommendations for test users using item-based collaborative filtering.


In [ ]:
# Item-based collaborative filtering recommendation function

# For each user
# - Collect the candidate items from the similarity index
# - Remove the items the user has already rated
# - Predict a rating for each candidate using a weighted average
# - Return the top-N cadidates by predicted rating

def generate_recommendations(user_id, similarity_index, user_train, top_n=10):
  """ Generate top-n item recommendations for a user using item based colaborative filtering.

  similarity_index : {item: {similar_item: cosine_similarity_score}}
  user_train : {item: rating} — items the user has already rated
  top_n : number of recommendations to return (fixed to 10)

  Returns a dict of {item_id: predicted_rating} sorted by predicted rating
  """
  if user_id not in user_train:
    return {}

  rated_items = user_train[user_id]
  predicted_ratings = {}

  # Collect all candidate items from the similarity index
  candidate_items = set()
  for rated_item in rated_items:
    if rated_item in similarity_index:
      candidate_items.update(similarity_index[rated_item].keys())

  # remove the items the user has already rated
  candidate_items -= set(rated_items.keys())

  # Predict the rating for each cadidate item using weighted average
  for candidate in candidate_items:
    numerator = 0.0
    denominator = 0.0
    for rated_item, user_rating in rated_items.items():
      sim = similarity_index.get(candidate, {}).get(rated_item, 0.0)
      if sim > 0:
        numerator += sim * user_rating
        denominator += sim
    if denominator > 0:
      predicted_ratings[candidate] = numerator / denominator

  # Return top-n sorted by predicted rating
  return dict(
      sorted(predicted_ratings.items(), key=lambda x: x[1], reverse=True)[:top_n]
  )

print("Recommendation function ready")

---
## Section 9 - Evalutaion
For each unseen item, the predicted rating is a weighted average of the user's existing ratings, weighted by how similar each rated item is to the item in question:


**Evaluation metrics:**
- **Precision@10** — of the top 10 recommendations, what fraction did the user actually rate ≥ 3 in the test set?
- **RMSE** — root mean squared error between predicted and actual ratings for overlapping items
- **MAE** — mean absolute error; more interpretable than RMSE on a 1-5 rating scale

In [ ]:
# Evaluation function
def run_evaluation(lsh_similarities, train_items=None, test_items=None):
  """Evaluate recommendations for 500 test users.
    Returns precision@10, rmse, and mae and also accepts options subsets"""

  if train_items is None:
    train_items = user_train_items
  if test_items is None:
    test_items = user_test_items

  eval_users = [u for u in test_items if u in train_items]
  EVAL_SAMPLE = min(500, len(eval_users))
  np.random.seed(42)
  sample_users = np.random.choice(eval_users, size=EVAL_SAMPLE, replace=False)

  precision_scores, rmse_errors, mae_errors = [], [], []

  for user_id in sample_users:
    recs = generate_recommendations(
        user_id, lsh_similarities, train_items, top_n=10
    )
    if not recs:
      continue
    test_set = test_items[user_id]

    # Precision@10
    hits = sum(
        1 for item in recs
        if item in test_set and test_set[item] >= 3.0
    )
    precision_scores.append(hits / 10)

    # RMSE and MAE (I dont have 100% confidence in these measurements)
    for item, predicted in recs.items():
      actual_key = str(item)
      if actual_key in test_set:
        actual = test_set[actual_key]
        rmse_errors.append((predicted - actual) ** 2)
        mae_errors.append(abs(predicted - actual))

  precision_at_10 = np.mean(precision_scores) if precision_scores else 0
  rmse = np.sqrt(np.mean(rmse_errors)) if rmse_errors else 0
  mae = np.mean(mae_errors) if mae_errors else 0
  print(f"  Evaluation : Precision@10={precision_at_10:.3f} | "
        f"RMSE={rmse:.3f} | MAE={mae:.3f}")

  return precision_at_10, rmse, mae

print("Recommendation and evaluation functions ready")

---
## Section 10 - Main Pipeline Loop

Calls each function defined in sections 4, 5, 7, 8 and 9 for all three NUM_HASHES configurations (50, 100, 200) automatically. Results are collected into a dictionary and printed into a summary table in section 10.

In [ ]:
# Main pipeline loop
# Calls run_minhash, run_lsh_banding, run_similarity, run_evaluation

results = {}

for NUM_HASHES in [50, 100, 200]:
  print(f"\n{'='*50}")
  print(f" NUM_HASHES = {NUM_HASHES}")
  print(f"\n{'='*50}")

  signatures, sig_time  = run_minhash(NUM_HASHES)
  candidate_pairs, NUM_BANDS = run_lsh_banding(signatures, NUM_HASHES)
  lsh_similarities, lsh_time = run_similarity(candidate_pairs)
  precision_at_10, rmse, mae = run_evaluation(lsh_similarities)

  results[NUM_HASHES] = {
      'precision': precision_at_10,
      'rmse': rmse,
      'mae': mae,
      'lsh_time': lsh_time,
      'candidates': len(candidate_pairs)
  }

# preserve the 200-hash index for the direct comparison cell
lsh_similarities_final = lsh_similarities
lsh_time_final = lsh_time

print("\nAll three configurations complete")

In [ ]:
# Brute-force timing comparison
# Computing all pairwise similarities would take too long
# Instead time it for a 1000 item sample and extrapolate

SAMPLE_SIZE = 1000
print(f"Timing brute-force on a sample of {SAMPLE_SIZE} items...")
sample_items = list(item_rating_vectors.keys())[:SAMPLE_SIZE]

start_brute = time.time()
brute_sample_similarities = defaultdict(dict)
sample_pair_count = 0

for i in range(len(sample_items)):
  for j in range(i + 1, len(sample_items)):
    item_a = sample_items[i]
    item_b = sample_items[j]
    sim = cosine_similarity(item_a, item_b, item_rating_vectors)
    if sim > 0:
      brute_sample_similarities[item_a][item_b] = sim
      brute_sample_similarities[item_b][item_a] = sim
    sample_pair_count += 1

brute_sample_time = time.time() - start_brute

# Extrapolate to the full dataset
total_pairs = len(item_rating_vectors) * (len(item_rating_vectors) - 1) / 2
sample_pairs = SAMPLE_SIZE * (SAMPLE_SIZE - 1) / 2
extrapolated_time = (brute_sample_time / sample_pairs) * total_pairs

print(f"\nBrute-force timing results:")
print(f"  Sample pairs computed   : {sample_pair_count:,} in {brute_sample_time:.1f}s")
print(f"  Total pairs (full)      : {total_pairs:,.0f}")
print(f"  Extrapolated full time  : {extrapolated_time/60:.1f} minutes")

In [ ]:
# Direct accuracy comparison: LSH vs brute-force on the same 2000-item subset
# N_SUBSETS = 1 runs once (fast)
# N_SUBSETS = 20 runs 20 random subsets and averages results (slow)

SUBSET_SIZE  = 2000
N_SUBSETS = 1

print(f"Running direct comparison: {N_SUBSETS} subsets of {SUBSET_SIZE} items...\n")

all_item_keys = list(item_rating_vectors.keys())
all_bf_p10, all_lsh_p10 = [], []
all_bf_time, all_lsh_time = [], []

for trial in range(N_SUBSETS):

  # select subset, fixed for the first trial, random after that
  if N_SUBSETS == 1:
    np.random.seed(42)
    subset_indices = np.random.choice(len(all_item_keys), size=SUBSET_SIZE, replace=False)
    subset_items   = [all_item_keys[i] for i in subset_indices]
  else:
    np.random.seed(trial)
    indices      = np.random.choice(len(all_item_keys), size=SUBSET_SIZE, replace=False)
    subset_items = [all_item_keys[i] for i in indices]

  subset_set = set(subset_items)
  subset_user_sets = {k: v for k, v in item_user_sets.items() if k in subset_set}
  subset_rating_vec = {k: v for k, v in item_rating_vectors.items() if k in subset_set}

  # filter train/test to subset items only
  subset_train = {
      u: {item: r for item, r in items.items() if item in subset_set}
      for u, items in user_train_items.items()
  }
  subset_train = {u: items for u, items in subset_train.items() if len(items) >= 2}

  subset_test = {
      u: {item: r for item, r in items.items() if item in subset_set}
      for u, items in user_test_items.items()
      if u in subset_train
  }
  subset_test = {u: items for u, items in subset_test.items() if len(items) >= 1}

  if not subset_test:
    print(f"  Trial {trial+1}, no valid test users")
    continue

  eval_users = list(subset_test.keys())
  sample_size = min(500, len(eval_users))
  np.random.seed(trial + 100)
  sample_users = np.random.choice(eval_users, size=sample_size, replace=False)

  # Brute-force on subset
  if N_SUBSETS == 1:
    print("Running BF on subset...")
  start_bf = time.time()
  brute_similarities = defaultdict(dict)

  for i in range(len(subset_items)):
    for j in range(i + 1, len(subset_items)):
      item_a = subset_items[i]
      item_b = subset_items[j]
      sim = cosine_similarity(item_a, item_b, subset_rating_vec)
      if sim > 0:
        brute_similarities[item_a][item_b] = sim
        brute_similarities[item_b][item_a] = sim

  brute_build_time = time.time() - start_bf
  if N_SUBSETS == 1:
    print(f"  Done in {brute_build_time:.1f}s\n")

  # LSH on same subset
  if N_SUBSETS == 1:
    print("Running LSH pipeline on subset...")
  start_lsh_sub = time.time()
  sub_sigs, _ = run_minhash(200, item_sets=subset_user_sets)
  sub_pairs, _ = run_lsh_banding(sub_sigs, 200)
  sub_sims, _ = run_similarity(sub_pairs, rating_vectors=subset_rating_vec)
  lsh_subset_time = time.time() - start_lsh_sub
  if N_SUBSETS == 1:
    print(f"  Done in {lsh_subset_time:.1f}s\n")

  # Evaluate both methods on the same subset
  bf_p10, bf_rmse, bf_mae = run_evaluation(brute_similarities, subset_train, subset_test)
  lsh_p10, _, _ = run_evaluation(sub_sims, subset_train, subset_test)

  # append results from test run
  all_bf_p10.append(bf_p10)
  all_lsh_p10.append(lsh_p10)
  all_bf_time.append(brute_build_time)
  all_lsh_time.append(lsh_subset_time)

# final test results
avg_bf_p10  = np.mean(all_bf_p10)
avg_lsh_p10 = np.mean(all_lsh_p10)
avg_bf_time  = np.mean(all_bf_time)
avg_lsh_time = np.mean(all_lsh_time)



In [ ]:
# Full results table

print("\n" + "="*50)
print("RESULTS TABLE")
print("="*50)
print(f"{'Method':<38} {'Precision@10':>12} {'RMSE':>8} {'MAE':>8} {'Candidates':>14} {'Runtime':>12}")
print("-"*50)
print(
  f"{'Brute-force CF (2000-item subset)':<38} {bf_p10:>12.3f} {bf_rmse:>8.3f} "
  f"{bf_mae:>8.3f} {'N/A':>14} {extrapolated_time/60:>12.1f}m"
)
for n, r in results.items():
  method = f"LSH CF ({n} hash functions)"
  print(
      f"{method:<38} {r['precision']:>12.3f} {r['rmse']:>8.3f} "
      f"{r['mae']:>8.3f} {r['candidates']:>14} {str(round(r['lsh_time'], 1)) + 's':>12}"
  )
print("="*50)
print("NOTE: Brute-force accuracy metrics computed on 2000-item subset,\n      LSH accuracy metrics computed on full dataset")
print("\nSpeedup vs brute-force:")
for n, r in results.items():
  print(f"  {n} hash functions: ~{extrapolated_time/r['lsh_time']:.0f}x faster")

In [ ]:
# Print a table showing just the results from the test subset

title = (
  "DIRECT COMPARISON (same 2000-item subset, same users)"
  if N_SUBSETS == 1
  else f"DIRECT COMPARISON (averaged over {N_SUBSETS} random subsets)"
)

precision_diff = avg_lsh_p10 / avg_bf_p10 if avg_bf_p10 > 0 else 0
runtime_speedup = avg_bf_time / avg_lsh_time if avg_lsh_time > 0 else 0

print(f"{'='*50}")
print(title)
print(f"{'='*50}")
print(f"{'Method':<30} {'Precision@10':>12} {'Runtime':>10}")
print(f"{'-'*50}")
print(f"{'Brute-force CF':<30} {avg_bf_p10:>12.3f} {f'{avg_bf_time:.1f}s':>10}")
print(f"{'LSH CF (200 hash functions)':<30} {avg_lsh_p10:>12.3f} {f'{avg_lsh_time:.1f}s':>10}")
print(f"{'Speedup':<30} {f'~{precision_diff:.0f}x':>12} {f'~{runtime_speedup:.0f}x':>10}")
print(f"{'='*50}")
print("Both approaches evaluated on same data")

if N_SUBSETS > 1:
  print(f"\nStandard deviation across {N_SUBSETS} subsets:")
  print(f"  Brute-force Precision@10 : {np.std(all_bf_p10):.4f}")
  print(f"  LSH Precision@10         : {np.std(all_lsh_p10):.4f}")

In [ ]:
# Run the scalability test for all three hash configurations and plot

print("Running single processor scalability test...")
print("This tests 4 problem sizes (25%, 50%, 75%, 100%) for each configuration\n")

scalability_results = {}

for n in [50, 100, 200]:
  print(f"Testing NUM_HASHES = {n}:")
  sizes, times = run_scalability_test(n)
  scalability_results[n] = (sizes, times)
  print()

plot_scalability(scalability_results)

---
## Section 10 — Weak Scalability (Google Cloud Dataproc)

Weak scalability measures whether runtime stays constant as both the problem size and number of workers increase. Ideal weak scalability means doubling the data and doubling the workers produces the same runtime.

| Workers | Items |
|---------|-------|
| 1 | 5,000 |
| 2 | 10,000 |
| 4 | 20,000 |
| 8 | 40,000 |

In [ ]:
%%writefile minhash_lsh_cloud.py
# lsh pipeline for Google Cloud Dataproc

import time
import sys
import json
import numpy as np
from collections import defaultdict
import subprocess

N_ITEMS = int(sys.argv[1]) if len(sys.argv) > 1 else 10000
N_WORKERS = int(sys.argv[2]) if len(sys.argv) > 2 else 1
NUM_HASHES = 200
ROWS_PER_BAND = 2

def minhash_signature(item_tuple):
  item_id, user_set = item_tuple
  user_ints = np.array(
      [user_to_int[u] for u in user_set if u in user_to_int],
      dtype=np.int64
  )
  if len(user_ints) == 0:
      return (item_id, np.full(NUM_HASHES, int(p), dtype=np.int64))
  hash_matrix = (np.outer(a, user_ints) + b[:, None]) % int(p)
  return (item_id, hash_matrix.min(axis=1))

def get_band_keys(item_tuple):
  """For one item generate all (band_key, item_id) pairs — runs in parallel"""
  item_id, sig = item_tuple
  pairs = []
  for band_idx in range(NUM_BANDS):
    s = band_idx * ROWS_PER_BAND
    band_slice = tuple(int(x) for x in sig[s:s + ROWS_PER_BAND])
    key = f"{band_idx}_{band_slice[0]}_{band_slice[1]}"
    pairs.append((key, item_id))
  return tuple(pairs)

def extract_pairs(bucket_tuple):
  """Extract all candidate pairs from one bucket"""
  _, items_in_bucket = bucket_tuple
  item_ids = [item_id for _, item_id in items_in_bucket]
  pairs = []
  if len(item_ids) > 1:
    for i in range(len(item_ids)):
      for j in range(i + 1, len(item_ids)):
        pairs.append(tuple(sorted([item_ids[i], item_ids[j]])))
  return pairs

if __name__ == "__main__":
  import dask.bag as db

  print(f"LSH Pipeline: {N_ITEMS} items, {N_WORKERS} workers")

  # Load pre processed training data from Google Cloud Storage
  subprocess.run(["gsutil", "cp", f"your-gcs-bucket-name", "item_user_sets.json"], check=True)

  with open("item_user_sets.json", "r") as f:
    item_user_sets_raw = json.load(f)

  item_user_sets = {
    k: set(v)
    for k, v in list(item_user_sets_raw.items())[:N_ITEMS]
  }
  print(f"Loaded {len(item_user_sets):,} item user sets from GCS")

  # user to integer mapping
  all_users = list({u for users in item_user_sets.values() for u in users})
  user_to_int = {user: idx for idx, user in enumerate(all_users)}


  # hash function parameters
  np.random.seed(42)
  p = 2**31 - 1
  a = np.random.randint(1, p, size=NUM_HASHES).astype(np.int64)
  b = np.random.randint(0, p, size=NUM_HASHES).astype(np.int64)

  start = time.time()

  # MinHash using Dask Bag with N_WORKERS partitions
  items_bag  = db.from_sequence(item_user_sets.items(), npartitions=N_WORKERS * 2)
  signatures = dict(
    items_bag
    .map(minhash_signature)
    .compute(scheduler='synchronous')
  )

  # LSH banding
  NUM_BANDS = NUM_HASHES // ROWS_PER_BAND

  # map each item to its band keys in parallel
  sigs_bag = db.from_sequence(list(signatures.items()), npartitions=8)
  band_pairs = sigs_bag.map(get_band_keys).flatten()

  # group by band key to form buckets
  grouped = band_pairs.groupby(lambda x: x[0])

  candidate_pairs = set(
      grouped
      .map(extract_pairs)
      .flatten()
      .compute(scheduler='synchronous')
    )

  elapsed = time.time() - start
  print(f"RESULT: items={N_ITEMS} workers={N_WORKERS} time={elapsed:.2f}s candidates={len(candidate_pairs)}")

In [ ]:
# Google Cloud Setup (Same as lab 3)

USERNAME = "your-username"

import os
os.environ["REGION"] = "australia1"
os.environ["ZONE"] = "australia-southeast1-c"
os.environ["PROJECT"] = f"data301-project-2026-{USERNAME}"
os.environ["CLUSTER"] = f"data301-project-2026-{USERNAME}-cluster"

print(f"Project : {os.environ['PROJECT']}")
print(f"Cluster : {os.environ['CLUSTER']}")

In [ ]:
# Authenticate and enable required APIs
!gcloud auth login
!gcloud config set project $PROJECT
!gcloud services enable dataproc.googleapis.com cloudresourcemanager.googleapis.com

# create a bucket
BUCKET_NAME = "your-gcs-bucket-name"
SERVICE_ACCOUNT = "your-service-account"

#!gcloud storage buckets create gs://{BUCKET_NAME} --location=us-central1

print("Using bucket:", BUCKET_NAME)

# give the service account access to the bucket
!gcloud storage buckets add-iam-policy-binding gs://{BUCKET_NAME} \
  --member=serviceAccount:{SERVICE_ACCOUNT} \
  --role=roles/storage.objectAdmin


In [ ]:
# Serialise and copy over item user set data

import json

item_user_sets_serialisable = {k: list(v) for k, v in item_user_sets.items()}

with open(f"{SAVE_DIR}/item_user_sets.json", "w") as f:
    json.dump(item_user_sets_serialisable, f)

!gsutil cp "{SAVE_DIR}/item_user_sets.json" gs://{BUCKET_NAME}/item_user_sets.json
print("Uploaded item_user_sets.json to GCS")

In [ ]:
# Only run if cluster breaks
!gcloud dataproc clusters delete $CLUSTER \
  --region=$REGION

In [ ]:
# Create the Dataproc cluster

!gcloud dataproc clusters create $CLUSTER \
  --region=$REGION \
  --master-machine-type n1-standard-2 \
  --master-boot-disk-size 50 \
  --worker-machine-type n1-standard-2 \
  --worker-boot-disk-size 50 \
  --num-workers=2 \
  --image-version 2.2-ubuntu22 \
  --max-age=60m \
  --public-ip-address

In [ ]:
# Weak scalability test runs
# Problem size doubles as workers double

# 1 worker 10,000 items
!gcloud dataproc jobs submit pyspark --cluster=$CLUSTER --region=$REGION \
  minhash_lsh_cloud.py -- 5000 1

# 2 workers 20,000 items
!gcloud dataproc jobs submit pyspark --cluster=$CLUSTER --region=$REGION \
  minhash_lsh_cloud.py -- 10000 2

# # 4 workers 40,000 items
!gcloud dataproc jobs submit pyspark --cluster=$CLUSTER --region=$REGION \
  minhash_lsh_cloud.py -- 20000 4

# # # 8 workers 80,000 items
!gcloud dataproc jobs submit pyspark --cluster=$CLUSTER --region=$REGION \
  minhash_lsh_cloud.py -- 40000 8

In [ ]:
# Weak scalability plot

weak_workers = [1, 2, 4, 8]
weak_items = [5000, 10000, 20000, 40000]
weak_times = [4.53, 8.55, 15.72, 30.65] # filled with the results


#RESULT: items=2000 workers=2 time=3.47s candidates=15866
#RESULT: items=1000 workers=1 time=1.22s candidates=2470
#RESULT: items=4000 workers=4 time=3.83s candidates=37253
#RESULT: items=8000 workers=8 time=6.98s candidates=157329

#RESULT: items=5000 workers=1 time=4.53s candidates=100386
#RESULT: items=10000 workers=1 time=8.55s candidates=261882
#RESULT: items=20000 workers=2 time=15.72s candidates=585983
#RESULT: items=40000 workers=4 time=30.65s candidates=1457853
#RESULT: items=80000 workers=8 time=31.30s candidates=2017716 (actually 40,946 items i think)


fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(weak_workers, weak_times,
      'o-', color='blue', linewidth=2, markersize=8,
      label='Actual runtime')

# ideal line
if weak_times[0] > 0:
  ax.axhline(y=weak_times[0], color='gray', linestyle='--',
            linewidth=1.5, label='Ideal (constant runtime)')

ax.set_xlabel('Number of Workers', fontsize=12)
ax.set_ylabel('Runtime (seconds)', fontsize=12)
ax.set_title('Weak Scalability — LSH Pipeline (Google Cloud)', fontsize=13)
ax.set_xticks(weak_workers)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

for w, n, t in zip(weak_workers, weak_items, weak_times):
  ax.annotate(f'{n:,} items', (w, t),
              textcoords='offset points', xytext=(15, 2), fontsize=9)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/scalability_weak.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {SAVE_DIR}/scalability_weak.png")